In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import optuna
import sklearn
import xgboost
import shap
import matplotlib.pyplot as plt
import joblib
import json

# Load data
df_qb = pd.read_csv("../data/processed/qb_combined_training.csv")
print(df_qb.shape)

# Features and target
ID_COLS = ["player_id", "player_display_name", "season"]
LEAK_COLS = ["target_fp_ppr", "target_games", "games",
             "fumble_recovery_own_pg", "pick", "round"]
X_qb = df_qb.drop(columns=ID_COLS + LEAK_COLS)
y_qb = df_qb["target_fp_ppr"]
strat_qb = df_qb["college_flag"]

# CV folds
sgkf_qb = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_indices_qb = list(sgkf_qb.split(X_qb, strat_qb, groups=df_qb["player_id"]))

for fold, (train_idx, val_idx) in enumerate(cv_indices_qb):
    train_college = strat_qb.iloc[train_idx].sum()
    val_college = strat_qb.iloc[val_idx].sum()
    print(f"Fold {fold+1} | Train: {len(train_idx)} rows ({train_college} college) "
          f"| Val: {len(val_idx)} rows ({val_college} college)")

# ── Optuna Objectives ──

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=50),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "max_features": trial.suggest_float("max_features", 0.2, 0.8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
    }

    scores = []
    for fold, (train_idx, val_idx) in enumerate(cv_indices_qb):
        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(X_qb.iloc[train_idx], y_qb.iloc[train_idx])
        preds = model.predict(X_qb.iloc[val_idx])
        rmse = np.sqrt(mean_squared_error(y_qb.iloc[val_idx], preds))
        scores.append(rmse)

        trial.report(rmse, fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return -np.mean(scores)

def objective_xgb(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
    }

    scores = []
    for fold, (train_idx, val_idx) in enumerate(cv_indices_qb):
        model = XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0,
                             early_stopping_rounds=50)
        model.fit(X_qb.iloc[train_idx], y_qb.iloc[train_idx],
                  eval_set=[(X_qb.iloc[val_idx], y_qb.iloc[val_idx])],
                  verbose=False)
        preds = model.predict(X_qb.iloc[val_idx])
        rmse = np.sqrt(mean_squared_error(y_qb.iloc[val_idx], preds))
        scores.append(rmse)

        trial.report(rmse, fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return -np.mean(scores)

# ── Run Studies ──

optuna.logging.set_verbosity(optuna.logging.WARNING)
pruner = optuna.pruners.MedianPruner(n_startup_trials=10)

study_rf = optuna.create_study(direction="maximize", study_name="qb_rf", pruner=pruner)
study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

study_xgb = optuna.create_study(direction="maximize", study_name="qb_xgb", pruner=pruner)
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

# ── Model Comparison ──

rf_rmse = -study_rf.best_value
xgb_rmse = -study_xgb.best_value

print("\n--- QB Model Comparison ---")
print(f"Random Forest RMSE: {rf_rmse:.4f}  (best trial: {study_rf.best_trial.number})")
print(f"XGBoost RMSE:       {xgb_rmse:.4f}  (best trial: {study_xgb.best_trial.number})")

# ── Refit Best Model on Full Data ──

if rf_rmse < xgb_rmse:
    best_model_qb = RandomForestRegressor(
        **study_rf.best_params, random_state=42, n_jobs=-1
    )
    model_name_qb = "rf"
    best_params_qb = study_rf.best_params
else:
    # refit xgb without early stopping for final model
    xgb_params = {k: v for k, v in study_xgb.best_params.items()}
    xgb_params["n_estimators"] = 1000
    best_model_qb = XGBRegressor(
        **xgb_params, random_state=42, n_jobs=-1, verbosity=0
    )
    model_name_qb = "xgb"
    best_params_qb = study_xgb.best_params

best_model_qb.fit(X_qb, y_qb)

# ── SHAP ──

explainer_qb = shap.TreeExplainer(best_model_qb)
shap_values_qb = explainer_qb.shap_values(X_qb)

shap.summary_plot(shap_values_qb, X_qb, show=False)
plt.title("QB SHAP Beeswarm - Direction and Magnitude")
plt.tight_layout()
plt.savefig(f"qb_{model_name_qb}_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()

shap.summary_plot(shap_values_qb, X_qb, plot_type="bar", show=False)
plt.title("QB SHAP Feature Importance (Mean |SHAP|)")
plt.tight_layout()
plt.savefig(f"qb_{model_name_qb}_shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()

# ── Save Artifacts ──

joblib.dump(best_model_qb, f"best_{model_name_qb}_qb_model.joblib")
joblib.dump(study_rf, "qb_study_rf.joblib")
joblib.dump(study_xgb, "qb_study_xgb.joblib")

feature_cols_qb = X_qb.columns.tolist()
with open(f"{model_name_qb}_qb_feature_cols.json", "w") as f:
    json.dump(feature_cols_qb, f)

metadata_qb = {
    "model_name": model_name_qb,
    "rf_rmse": round(rf_rmse, 4),
    "xgb_rmse": round(xgb_rmse, 4),
    "best_params": best_params_qb,
    "n_features": len(feature_cols_qb),
    "n_samples": len(X_qb),
    "college_pct": round(strat_qb.mean(), 3),
    "rf_trials": len(study_rf.trials),
    "xgb_trials": len(study_xgb.trials),
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "optuna_version": optuna.__version__,
}
with open("qb_model_metadata.json", "w") as f:
    json.dump(metadata_qb, f, indent=2)

print(f"\nQB {model_name_qb.upper()} model saved")
print(f"Best params: {best_params_qb}")

(883, 26)
Fold 1 | Train: 706 rows (80 college) | Val: 177 rows (21 college)
Fold 2 | Train: 705 rows (81 college) | Val: 178 rows (20 college)
Fold 3 | Train: 706 rows (81 college) | Val: 177 rows (20 college)
Fold 4 | Train: 708 rows (81 college) | Val: 175 rows (20 college)
Fold 5 | Train: 707 rows (81 college) | Val: 176 rows (20 college)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]


--- QB Model Comparison ---
Random Forest RMSE: 4.5715  (best trial: 1)
XGBoost RMSE:       4.5717  (best trial: 6)

QB RF model saved
Best params: {'n_estimators': 450, 'max_depth': 24, 'max_features': 0.523896085296933, 'min_samples_leaf': 13, 'min_samples_split': 17}
